# LeetCode 841 — Keys and Rooms bằng Python

Link đề: https://leetcode.com/problems/keys-and-rooms/ · Độ khó: **Medium** · Chủ đề: Depth-First Search, Breadth-First Search, Graph

Skeleton LeetCode:

```python
class Solution:
    def canVisitAllRooms(self, rooms: List[List[int]]) -> bool:
        ...
```

Ghi chú: bài có **đáp án duy nhất** (`True`/`False`). Đây là bài **đồ thị có hướng**: phòng là đỉnh, chìa khoá là cạnh `i → j` (trong phòng `i` có chìa mở phòng `j`). Câu hỏi: từ đỉnh `0` có **tới được mọi đỉnh** không? Notebook dùng bao đóng bắc cầu Floyd–Warshall làm oracle độc lập, và vét cạn mọi đồ thị có hướng nhỏ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: `n` phòng đánh số `0..n−1`, chỉ phòng `0` mở sẵn; `rooms[i]` là các chìa tìm được trong phòng `i` (mỗi chìa mở đúng một phòng); hỏi có thể vào **mọi** phòng không.
2. Nắm các công cụ Python: danh sách kề (`rooms` chính là danh sách kề!), tập `set` đánh dấu đã thăm, `collections.deque` (BFS), ngăn xếp bằng `list`, đệ quy và giới hạn độ sâu, bitmask bằng số nguyên.
3. Đi từ **lặp tới khi hết thay đổi (bao đóng, brute force)** → **DFS đệ quy** → **BFS (và DFS bằng ngăn xếp)** → **bao đóng bằng bitmask (mỗi hàng là một số nguyên)**.
4. Hiểu mấu chốt: đây là bài **khả năng tới được (reachability)** trên đồ thị **có hướng**; cần một tập `visited`, và đáp án là `len(visited) == n`. Chìa khoá chỉ mở cửa **một chiều** nên **không** dùng Union-Find (kết nối vô hướng).
5. Tránh các lỗi kinh điển: quên `visited` (lặp vô hạn khi có chu trình), không đánh dấu phòng `0`, coi đồ thị là vô hướng, đếm sai (`n − 1` thay `n`), và đệ quy quá sâu với chuỗi 1000 phòng.

## 2. Đề bài & ví dụ

Có `n` phòng đánh số từ `0` đến `n − 1`, **tất cả bị khoá trừ phòng `0`**. Mục tiêu là vào được **mọi** phòng, nhưng không thể vào phòng bị khoá nếu chưa có chìa của nó.

Khi vào một phòng, ta có thể tìm thấy một tập **chìa khoá phân biệt**; mỗi chìa có ghi một số, cho biết nó mở phòng nào, và ta mang theo mọi chìa đó để mở các phòng khác.

Cho mảng `rooms` với `rooms[i]` là tập chìa có trong phòng `i`. Trả về `true` nếu có thể vào **tất cả** phòng, ngược lại `false`.

**Ví dụ 1:** `rooms = [[1],[2],[3],[]]` → `true`. Vào phòng `0` lấy chìa `1`; vào phòng `1` lấy chìa `2`; vào phòng `2` lấy chìa `3`; vào phòng `3`.

**Ví dụ 2:** `rooms = [[1,3],[3,0,1],[2],[0]]` → `false`. Không thể vào phòng `2` vì chìa duy nhất mở nó nằm ngay trong phòng `2`.

**Ràng buộc:**

- `n == rooms.length`, `2 <= n <= 1000`
- `0 <= rooms[i].length <= 1000`, `1 <= sum(rooms[i].length) <= 3000`
- `0 <= rooms[i][j] < n`; các giá trị trong mỗi `rooms[i]` **phân biệt**.

In [1]:
import random
import sys
import time
from collections import deque
from itertools import product
from typing import List

ex1 = [[1], [2], [3], []]
ex2 = [[1, 3], [3, 0, 1], [2], [0]]
for rooms in (ex1, ex2):
    edges = [(i, j) for i, keys in enumerate(rooms) for j in keys]
    print(f"n={len(rooms)} phòng, {len(edges)} chìa (cạnh có hướng): {edges}")

n=4 phòng, 3 chìa (cạnh có hướng): [(0, 1), (1, 2), (2, 3)]
n=4 phòng, 7 chìa (cạnh có hướng): [(0, 1), (0, 3), (1, 3), (1, 0), (1, 1), (2, 2), (3, 0)]


## 3. Kiến thức nền

### 3.1 Danh sách kề và đồ thị có hướng

Một **đồ thị** gồm các **đỉnh** (ở đây là phòng) và các **cạnh** (chìa khoá). Cạnh **có hướng** `i → j`: từ phòng `i` ta lấy được chìa mở phòng `j`, **không có nghĩa** ngược lại (phòng `j` chưa chắc có chìa mở phòng `i`). `rooms[i]` chính là **danh sách các đỉnh kề** của `i` — cách biểu diễn gọi là **danh sách kề (adjacency list)**.

In [2]:
graph = ex2
for u, adj in enumerate(graph):
    print(f"phòng {u} có chìa mở các phòng {adj}")
print("có cạnh 1->3 và 3->0:", 3 in graph[1], 0 in graph[3], "| 0->2? ", 2 in graph[0], "| ai mở được phòng 2? chỉ phòng", [u for u, a in enumerate(graph) if 2 in a])

phòng 0 có chìa mở các phòng [1, 3]
phòng 1 có chìa mở các phòng [3, 0, 1]
phòng 2 có chìa mở các phòng [2]
phòng 3 có chìa mở các phòng [0]
có cạnh 1->3 và 3->0: True True | 0->2?  False | ai mở được phòng 2? chỉ phòng [2]


### 3.2 Khả năng tới được (reachability) và tập đã thăm `visited`

Phòng `v` **tới được** từ `0` nếu có dãy cạnh `0 → a → b → … → v`. Để tìm mọi đỉnh tới được, ta **duyệt** đồ thị từ `0` và ghi nhớ những đỉnh đã thăm trong một `set` (kiểm tra thuộc tập `O(1)`); nhờ `visited` mỗi đỉnh chỉ xử lý **một lần** dù có nhiều chìa dẫn tới nó hoặc có chu trình.

### 3.3 DFS và BFS

- **DFS** (duyệt theo chiều sâu): đi hết một nhánh rồi mới quay lui; viết bằng **đệ quy** hoặc bằng **ngăn xếp** (`list` với `append`/`pop`).
- **BFS** (duyệt theo chiều rộng): xử lý theo lớp, dùng **hàng đợi** `deque` (`append`/`popleft`).

Cả hai thăm đúng cùng một tập đỉnh (mọi đỉnh tới được); chỉ khác thứ tự.

In [3]:
q = deque([1, 2, 3]); q.append(4)
print("queue:", q.popleft(), q.popleft(), list(q))
st = [1, 2, 3]; st.append(4)
print("stack:", st.pop(), st.pop(), st)
seen = set()
for x in [0, 1, 1, 0, 2]:
    if x in seen:
        print(f"{x} đã thăm -> bỏ qua")
    else:
        seen.add(x)
print("visited =", seen)

queue: 1 2 [3, 4]
stack: 4 3 [1, 2]
1 đã thăm -> bỏ qua
0 đã thăm -> bỏ qua
visited = {0, 1, 2}


### 3.4 Giới hạn độ sâu đệ quy

Python mặc định chỉ cho đệ quy sâu khoảng **1000** tầng. Đồ thị có thể là một **chuỗi** `0 → 1 → 2 → … → n−1` với `n = 1000`; DFS đệ quy sẽ đi sâu `n` tầng (cộng thêm vài tầng của hàm gọi) → có thể `RecursionError`. BFS hoặc DFS bằng ngăn xếp không bị giới hạn này.

In [4]:
print("giới hạn mặc định:", sys.getrecursionlimit())
chain = [[i + 1] for i in range(999)] + [[]]       # 1000 phòng nối thành chuỗi
print("chuỗi:", len(chain), "phòng, phòng cuối có", len(chain[-1]), "chìa")


def depth(k):
    return 0 if k == 0 else 1 + depth(k - 1)


try:
    depth(1000)
    print("depth(1000) vẫn chạy (sát giới hạn)")
except RecursionError:
    print("depth(1000) -> RecursionError")

giới hạn mặc định: 1000
chuỗi: 1000 phòng, phòng cuối có 0 chìa
depth(1000) -> RecursionError


### 3.5 Bitmask: tập đỉnh là một số nguyên

Với `n ≤ 1000` đỉnh, một **tập đỉnh** có thể là một số nguyên lớn: bit `j` bật nghĩa là `j` thuộc tập. Phép hợp tập là `|`, kiểm tra thuộc `(mask >> j) & 1`, đếm phần tử `mask.bit_count()`. Số nguyên Python có độ dài tuỳ ý nên hợp hai tập `1000` đỉnh chỉ là một phép `|` trên ~128 byte.

In [5]:
a = (1 << 0) | (1 << 3) | (1 << 5)
b = (1 << 3) | (1 << 4)
print(f"a={a:06b} b={b:06b} hợp={a | b:06b} giao={a & b:06b} | số phần tử hợp:", (a | b).bit_count(), "| 5 thuộc a?", bool(a >> 5 & 1))

a=101001 b=011000 hợp=111001 giao=001000 | số phần tử hợp: 4 | 5 thuộc a? True


## 4. Ý tưởng nền tảng / trực giác

Bắt đầu với chìa của phòng `0` (phòng `0` luôn mở). Mỗi lần ta **vào** một phòng, ta thu thêm chìa; mỗi chìa có thể **mở** một phòng mới. Ta tiếp tục cho đến khi **không còn phòng mới nào mở được**. Khi đó tập phòng đã vào chính là tập các phòng **tới được**; đáp án `True` nếu tập này gồm đủ `n` phòng.

Tính tay ví dụ 2: `[[1,3],[3,0,1],[2],[0]]`.

- Vào phòng `0`: chìa `{1, 3}` → mở được phòng `1` và `3`.
- Vào phòng `1`: chìa `{3, 0, 1}`; không có chìa mới ngoài `{0, 1, 3}`.
- Vào phòng `3`: chìa `{0}`; vẫn không mới.
- Đã vào `{0, 1, 3}` (3 phòng), còn phòng `2` mà chìa `2` chỉ nằm trong phòng `2` → không bao giờ mở được → `False`.

Hình dung: từ `0` ta "lan" như vết dầu qua các cạnh có hướng; phòng nào vết dầu không tới là phòng bị khoá vĩnh viễn.

In [6]:
def trace_simple(rooms):
    opened = {0}
    keys = set(rooms[0])
    print("vào phòng 0, có chìa:", sorted(keys))
    changed = True
    while changed:
        changed = False
        for k in sorted(keys - opened):
            opened.add(k)
            keys |= set(rooms[k])
            print(f"mở phòng {k}, thu thêm chìa {rooms[k]} -> tổng chìa {sorted(keys)}")
            changed = True
    print("đã vào:", sorted(opened), "| phòng chưa vào:", sorted(set(range(len(rooms))) - opened))
    return len(opened) == len(rooms)


print(trace_simple(ex2))
print(trace_simple(ex1))

vào phòng 0, có chìa: [1, 3]
mở phòng 1, thu thêm chìa [3, 0, 1] -> tổng chìa [0, 1, 3]
mở phòng 3, thu thêm chìa [0] -> tổng chìa [0, 1, 3]
đã vào: [0, 1, 3] | phòng chưa vào: [2]
False
vào phòng 0, có chìa: [1]
mở phòng 1, thu thêm chìa [2] -> tổng chìa [1, 2]
mở phòng 2, thu thêm chìa [3] -> tổng chìa [1, 2, 3]
mở phòng 3, thu thêm chìa [] -> tổng chìa [1, 2, 3]
đã vào: [0, 1, 2, 3] | phòng chưa vào: []
True


## 5. Cách 1 — Brute Force: lặp tới khi tập phòng đã mở không đổi

**Ý tưởng:** giữ tập `opened` các phòng đã vào (ban đầu `{0}`). Lặp: với **mọi** phòng đã vào, thêm mọi phòng mà chìa của nó mở; nếu tập không đổi thì dừng. Đây là cách tính **điểm bất động** (fixed point) đơn giản nhất, không cần cấu trúc duyệt.

In [7]:
def can_visit_fixed_point(rooms):
    n = len(rooms)
    opened = {0}
    rounds = 0
    while True:
        rounds += 1
        new = set(opened)
        for u in opened:
            new.update(rooms[u])
        if new == opened:
            break
        opened = new
    can_visit_fixed_point.rounds = rounds
    return len(opened) == n


print(can_visit_fixed_point(ex1), can_visit_fixed_point(ex2))
print("số vòng lặp ở ví dụ 1:", (can_visit_fixed_point(ex1), can_visit_fixed_point.rounds)[1])
assert can_visit_fixed_point(ex1) is True and can_visit_fixed_point(ex2) is False

True False
số vòng lặp ở ví dụ 1: 4


**Độ phức tạp:** mỗi vòng duyệt mọi chìa của mọi phòng đã vào: `O(n + E)` (với `E` là tổng số chìa); số vòng tối đa `n` (chuỗi dài: mỗi vòng thêm đúng một phòng) nên xấu nhất `O(n · (n + E))`; với giới hạn đề (`n ≤ 1000`, `E ≤ 3000`) khoảng `4·10^6`. Bộ nhớ `O(n)`.

**Nút thắt:** vòng sau lại xử lý lại cả các phòng đã xử lý ở vòng trước. Chỉ cần xử lý phòng **mới** vừa mở: đó là DFS/BFS.

## 6. Ý tưởng thuật toán / bước đột phá

Mỗi phòng chỉ cần được **xử lý đúng một lần**: lúc nó được mở lần đầu, ta xem chìa của nó, và với mỗi chìa mở phòng **chưa thăm** thì thăm phòng đó (đánh dấu `visited` ngay). Hai cách tổ chức "việc cần làm":

- **Đệ quy (DFS):** `dfs(u)`: đánh dấu `u`; với mỗi chìa `k` trong `rooms[u]` chưa thăm, `dfs(k)`.
- **Hàng đợi/ngăn xếp (BFS / DFS không đệ quy):** đẩy phòng `0`; lấy ra một phòng, đẩy các phòng mới mở được (đã đánh dấu ngay khi đẩy).

Cuối cùng so `len(visited)` với `n`. Tổng thời gian `O(n + E)` vì mỗi phòng và mỗi chìa được xét đúng một lần.

## 7. Các cách tiếp theo

Ngoài Cách 1 (mục 5), ta có thêm 3 cách: **Cách 2** DFS đệ quy, **Cách 3** BFS (và DFS ngăn xếp), **Cách 4** bao đóng bằng bitmask.

### Cách 2 — DFS đệ quy

**Giải thích từng đoạn:**

1. `visited = set()`.
2. `dfs(u)`: nếu `u` đã thăm thì thôi; ngược lại thêm `u` vào `visited`, rồi gọi `dfs(k)` cho mọi chìa `k` trong `rooms[u]`.
3. Gọi `dfs(0)`; trả `len(visited) == n`.

Kiểm tra `visited` **trước** khi gọi đệ quy (hoặc ngay đầu hàm) để chặn chu trình.

In [8]:
def can_visit_dfs(rooms):
    visited = set()

    def dfs(u):
        if u in visited:
            return
        visited.add(u)
        for k in rooms[u]:
            dfs(k)

    dfs(0)
    return len(visited) == len(rooms)


print(can_visit_dfs(ex1), can_visit_dfs(ex2))
assert can_visit_dfs(ex1) is True and can_visit_dfs(ex2) is False

True False


In [9]:
# Code nộp LeetCode cho Cách 2
class SolutionDFS:
    def canVisitAllRooms(self, rooms: List[List[int]]) -> bool:
        sys.setrecursionlimit(10000)             # chuỗi 1000 phòng sâu ~1000 tầng
        visited = set()

        def dfs(u: int) -> None:
            if u in visited:
                return
            visited.add(u)
            for k in rooms[u]:
                dfs(k)

        dfs(0)
        return len(visited) == len(rooms)

**Độ phức tạp:** `O(n + E)` thời gian; bộ nhớ `O(n)` cho `visited` và call stack (tới `n` tầng với đồ thị chuỗi).

### Cách 3 — BFS bằng hàng đợi (và DFS bằng ngăn xếp)

**Giải thích từng đoạn:**

1. `visited = {0}`, `queue = deque([0])`.
2. Lấy `u = queue.popleft()`; với mỗi chìa `k` trong `rooms[u]`: nếu `k` chưa thăm thì **đánh dấu ngay** (`visited.add(k)`) rồi đẩy vào hàng đợi.
3. Khi hàng đợi rỗng, trả `len(visited) == n`.

Đánh dấu **lúc đưa vào hàng đợi** (không phải lúc lấy ra) để một phòng không bị đẩy nhiều lần. Đổi `popleft()` thành `pop()` (và `deque` thành `list`) ta có DFS dùng ngăn xếp.

In [10]:
def can_visit_bfs(rooms):
    visited = {0}
    queue = deque([0])
    while queue:
        u = queue.popleft()
        for k in rooms[u]:
            if k not in visited:
                visited.add(k)                  # đánh dấu NGAY khi đưa vào hàng đợi
                queue.append(k)
    return len(visited) == len(rooms)


def can_visit_stack(rooms):
    visited = {0}
    stack = [0]
    while stack:
        u = stack.pop()                         # LIFO thay cho FIFO
        for k in rooms[u]:
            if k not in visited:
                visited.add(k)
                stack.append(k)
    return len(visited) == len(rooms)


for fn in (can_visit_bfs, can_visit_stack):
    assert fn(ex1) is True and fn(ex2) is False
print("BFS và DFS-ngăn-xếp khớp 2 ví dụ")

BFS và DFS-ngăn-xếp khớp 2 ví dụ


In [11]:
# Code nộp LeetCode cho Cách 3 (đáp án chính)
from collections import deque


class SolutionBFS:
    def canVisitAllRooms(self, rooms: List[List[int]]) -> bool:
        visited = {0}
        queue = deque([0])
        while queue:
            u = queue.popleft()
            for k in rooms[u]:
                if k not in visited:
                    visited.add(k)
                    queue.append(k)
        return len(visited) == len(rooms)

**Độ phức tạp:** `O(n + E)` thời gian; bộ nhớ `O(n)`; **không** có nguy cơ `RecursionError`.

### Cách 4 — Bao đóng bằng bitmask

**Ý tưởng khác:** biểu diễn **tập đỉnh tới được** bằng một số nguyên `reach`, và lặp "mở rộng" bằng phép `|` trên cả tập cùng lúc. Với mỗi phòng `u` lưu `adj[u]` = bitmask các phòng mà chìa của `u` mở. Khởi đầu `reach = 1` (chỉ phòng `0`). Mỗi vòng: `new = reach` rồi với mọi `u` thuộc `reach`, `new |= adj[u]`; lặp tới khi `new == reach`. Phép lặp trên các bit bật dùng `x & -x` để lấy bit thấp nhất.

In [12]:
def can_visit_bitmask(rooms):
    n = len(rooms)
    adj = [sum(1 << k for k in keys) for keys in rooms]       # adj[u]: bitmask các phòng mở được từ u
    reach, frontier = 1, 1                                    # reach: tập đã tới; frontier: phần mới chưa xử lý
    while frontier:
        new = 0
        f = frontier
        while f:
            low = f & -f                                      # bit thấp nhất đang bật
            new |= adj[low.bit_length() - 1]
            f ^= low
        frontier = new & ~reach                               # chỉ giữ các phòng MỚI
        reach |= new
    return reach.bit_count() == n


print(can_visit_bitmask(ex1), can_visit_bitmask(ex2))
assert can_visit_bitmask(ex1) is True and can_visit_bitmask(ex2) is False

True False


In [13]:
# Code nộp LeetCode cho Cách 4
class SolutionBitmask:
    def canVisitAllRooms(self, rooms: List[List[int]]) -> bool:
        n = len(rooms)
        adj = [sum(1 << k for k in keys) for keys in rooms]
        reach = frontier = 1
        while frontier:
            new, f = 0, frontier
            while f:
                low = f & -f
                new |= adj[low.bit_length() - 1]
                f ^= low
            frontier = new & ~reach
            reach |= new
        return bin(reach).count("1") == n

**Độ phức tạp:** mỗi phòng được xử lý đúng một lần (khi xuất hiện trong `frontier`) với một phép `|` trên số nguyên `n` bit (`~n/64` từ máy): `O(n · n/64 + E)`; bộ nhớ `O(n²/64)` cho `adj`. Hay ở chỗ cho thấy **bao đóng bắc cầu** có thể tính bằng phép toán bit; với `n ≤ 1000` nó vẫn rất nhanh.

## 8. TRACE từng bước

**Trace Cách 3 (BFS)** trên ví dụ 2 `[[1,3],[3,0,1],[2],[0]]`:

| bước | lấy ra | các chìa | chưa thăm → đẩy vào | `visited` sau | hàng đợi sau |
|---|---|---|---|---|---|
| khởi tạo | — | — | — | `{0}` | `[0]` |
| 1 | `0` | `1, 3` | `1`, `3` | `{0, 1, 3}` | `[1, 3]` |
| 2 | `1` | `3, 0, 1` | (không có: cả ba đã thăm) | `{0, 1, 3}` | `[3]` |
| 3 | `3` | `0` | (không có) | `{0, 1, 3}` | `[]` |

Hàng đợi rỗng, `|visited| = 3 ≠ 4` → `False`. Hàm trace in từng bước, kể cả ví dụ 1:

In [14]:
def trace_bfs(rooms):
    visited = {0}
    queue = deque([0])
    step = 0
    print(f"khởi tạo: visited={sorted(visited)}, hàng đợi={list(queue)}")
    while queue:
        u = queue.popleft()
        step += 1
        pushed = []
        for k in rooms[u]:
            if k not in visited:
                visited.add(k)
                queue.append(k)
                pushed.append(k)
        print(f"bước {step}: lấy {u}, chìa {rooms[u]} -> đẩy {pushed if pushed else '-'} | visited={sorted(visited)} | hàng đợi={list(queue)}")
    print(f"=> đã thăm {len(visited)}/{len(rooms)} phòng ->", len(visited) == len(rooms))
    return len(visited) == len(rooms)


assert trace_bfs(ex2) is False
print()
assert trace_bfs(ex1) is True

khởi tạo: visited=[0], hàng đợi=[0]
bước 1: lấy 0, chìa [1, 3] -> đẩy [1, 3] | visited=[0, 1, 3] | hàng đợi=[1, 3]
bước 2: lấy 1, chìa [3, 0, 1] -> đẩy - | visited=[0, 1, 3] | hàng đợi=[3]
bước 3: lấy 3, chìa [0] -> đẩy - | visited=[0, 1, 3] | hàng đợi=[]
=> đã thăm 3/4 phòng -> False

khởi tạo: visited=[0], hàng đợi=[0]
bước 1: lấy 0, chìa [1] -> đẩy [1] | visited=[0, 1] | hàng đợi=[1]
bước 2: lấy 1, chìa [2] -> đẩy [2] | visited=[0, 1, 2] | hàng đợi=[2]
bước 3: lấy 2, chìa [3] -> đẩy [3] | visited=[0, 1, 2, 3] | hàng đợi=[3]
bước 4: lấy 3, chìa [] -> đẩy - | visited=[0, 1, 2, 3] | hàng đợi=[]
=> đã thăm 4/4 phòng -> True


**Trace Cách 2 (DFS đệ quy)** in cây gọi đệ quy thụt lề (ví dụ 2 và một đồ thị có chu trình `0 → 1 → 2 → 0`):

In [15]:
def trace_dfs(rooms):
    visited = set()

    def dfs(u, depth):
        pad = "  " * depth
        if u in visited:
            print(f"{pad}phòng {u}: đã thăm -> bỏ qua")
            return
        visited.add(u)
        print(f"{pad}vào phòng {u}, chìa {rooms[u]}")
        for k in rooms[u]:
            dfs(k, depth + 1)

    dfs(0, 0)
    print("=> đã thăm", sorted(visited), "->", len(visited) == len(rooms))
    return len(visited) == len(rooms)


assert trace_dfs(ex2) is False
print()
assert trace_dfs([[1], [2], [0]]) is True

vào phòng 0, chìa [1, 3]
  vào phòng 1, chìa [3, 0, 1]
    vào phòng 3, chìa [0]
      phòng 0: đã thăm -> bỏ qua
    phòng 0: đã thăm -> bỏ qua
    phòng 1: đã thăm -> bỏ qua
  phòng 3: đã thăm -> bỏ qua
=> đã thăm [0, 1, 3] -> False

vào phòng 0, chìa [1]
  vào phòng 1, chìa [2]
    vào phòng 2, chìa [0]
      phòng 0: đã thăm -> bỏ qua
=> đã thăm [0, 1, 2] -> True


## 9. Code nộp LeetCode

Chọn **BFS** (không đệ quy, `O(n + E)`). Giữ nguyên `class Solution`, `canVisitAllRooms(self, rooms)`:

In [16]:
from collections import deque
from typing import List


class Solution:
    def canVisitAllRooms(self, rooms: List[List[int]]) -> bool:
        visited = {0}                       # phòng 0 luôn mở
        queue = deque([0])
        while queue:
            u = queue.popleft()
            for k in rooms[u]:              # mỗi chìa mở một phòng k
                if k not in visited:
                    visited.add(k)          # đánh dấu ngay khi đưa vào hàng đợi
                    queue.append(k)
        return len(visited) == len(rooms)   # vào được mọi phòng?


print(Solution().canVisitAllRooms([[1], [2], [3], []]))                # True
print(Solution().canVisitAllRooms([[1, 3], [3, 0, 1], [2], [0]]))      # False
assert Solution().canVisitAllRooms(ex1) is True and Solution().canVisitAllRooms(ex2) is False

True
False


## 10. Vì sao đúng

**Định lý.** Khi thuật toán dừng, `visited` đúng bằng tập các phòng tới được từ `0` (qua các cạnh có hướng).

*Chứng minh.* (⊆) Mỗi phòng được thêm vào `visited` hoặc là phòng `0` hoặc là `k` với `k ∈ rooms[u]` cho một `u` đã thuộc `visited` (đã tới được); quy nạp theo thứ tự thêm, mọi phần tử của `visited` tới được từ `0`. (⊇) Giả sử có phòng `v` tới được từ `0` nhưng `v ∉ visited` khi dừng; lấy đường đi `0 = p_0 → p_1 → … → p_m = v` và gọi `p_t` là phòng **đầu tiên** trên đường không thuộc `visited` (`t ≥ 1`, vì `p_0 = 0 ∈ visited`). Khi đó `p_{t−1} ∈ visited`. Mọi phòng trong `visited` đã được lấy ra khỏi hàng đợi và xét **hết** chìa của nó (hàng đợi rỗng khi dừng, nên mọi phần tử từng vào hàng đợi đã được xử lý). Do đó chìa `p_t ∈ rooms[p_{t−1}]` đã được xét, và vì `p_t ∉ visited` nên đã được thêm — mâu thuẫn. ∎

**Mỗi phòng vào hàng đợi tối đa một lần** (đánh dấu ngay khi đẩy), nên số lần lấy ra `≤ n` và tổng số chìa được xét `≤ E`: `O(n + E)`.

**Vì sao không dùng Union-Find (kết nối vô hướng)?** DSU trả lời "hai đỉnh có cùng thành phần liên thông **vô hướng** không". Nhưng chìa chỉ mở **một chiều**: ví dụ `[[], [0]]`: có cạnh `1 → 0` nên 0 và 1 cùng thành phần vô hướng, mà từ phòng `0` (không có chìa) không vào được phòng `1` → đáp án `False`; DSU báo `True`. Ta minh hoạ ở mục 13.

Ta kiểm chứng bằng oracle độc lập **bao đóng bắc cầu Floyd–Warshall** và **vét cạn mọi đồ thị có hướng** nhỏ (mục 12).

## 11. Time / Space Complexity

Đặt `n` là số phòng, `E` là tổng số chìa.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Lặp tới khi hết thay đổi | `O(n · (n + E))` | `O(n)` | oracle đơn giản |
| 2. DFS đệ quy | `O(n + E)` | `O(n)` (call stack tới `n`) | nâng `setrecursionlimit` |
| 3. BFS / DFS ngăn xếp | `O(n + E)` | `O(n)` | **đáp án chính** |
| 4. Bitmask | `O(n²/64 + E)` | `O(n²/64)` | bao đóng bằng `|` |

Đo thực tế ở giới hạn lớn (`n = 1000`, `E = 3000`) và các hình dạng đồ thị khác nhau:

In [17]:
def random_rooms(rng, n, extra):
    """n phòng; tạo chuỗi bảo đảm có thể vào hết hoặc ngẫu nhiên tuỳ extra; mỗi phòng các chìa phân biệt."""
    rooms = [set() for _ in range(n)]
    for _ in range(extra):
        u, k = rng.randrange(n), rng.randrange(n)
        rooms[u].add(k)
    return [sorted(s) for s in rooms]


def bench(fn, rooms, repeat=5):
    best = 1e9
    for _ in range(repeat):
        t0 = time.perf_counter()
        fn(rooms)
        best = min(best, time.perf_counter() - t0)
    return best


rng = random.Random(841)
chain = [[i + 1] for i in range(999)] + [[]]
star = [list(range(1, 1000))] + [[] for _ in range(999)]
rand = random_rooms(rng, 1000, 3000)
for name, g in [("chuỗi 1000", chain), ("sao 1000", star), ("ngẫu nhiên 3000 chìa", rand)]:
    old = sys.getrecursionlimit()
    sys.setrecursionlimit(5000)
    res = [fn(g) for fn in (can_visit_fixed_point, can_visit_dfs, can_visit_bfs, can_visit_stack, can_visit_bitmask)]
    sys.setrecursionlimit(old)
    assert len(set(res)) == 1
    print(f"{name:22s} -> {res[0]!s:5}", end=" | ")
    sys.setrecursionlimit(5000)
    print(" ".join(f"{n_} {bench(fn, g) * 1000:7.3f} ms" for n_, fn in [("fixed", can_visit_fixed_point), ("dfs", can_visit_dfs), ("bfs", can_visit_bfs), ("stack", can_visit_stack), ("bits", can_visit_bitmask)]))
    sys.setrecursionlimit(old)

chuỗi 1000             -> True  | 

fixed  36.231 ms dfs   0.151 ms bfs   0.089 ms stack   0.091 ms bits   0.522 ms
sao 1000               -> True  | fixed   0.080 ms dfs   0.144 ms bfs   0.141 ms stack   0.133 ms bits   0.659 ms
ngẫu nhiên 3000 chìa   -> False | fixed   0.700 ms dfs   0.269 ms bfs   0.186 ms stack   0.176 ms bits   0.792 ms


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề, (B) edge case, (C) phản ví dụ: cài đặt sai phải bị bắt, (D) vét cạn mọi đồ thị có hướng nhỏ và oracle **Floyd–Warshall**, (E) ngẫu nhiên có seed, (F) giới hạn lớn nhất.

### A. Ví dụ của đề

In [18]:
IMPLS = {
    "fixed_point": can_visit_fixed_point,
    "dfs": can_visit_dfs,
    "bfs": can_visit_bfs,
    "stack": can_visit_stack,
    "bitmask": can_visit_bitmask,
    "Solution": lambda r: Solution().canVisitAllRooms([list(x) for x in r]),
    "SolutionDFS": lambda r: SolutionDFS().canVisitAllRooms([list(x) for x in r]),
    "SolutionBFS": lambda r: SolutionBFS().canVisitAllRooms([list(x) for x in r]),
    "SolutionBitmask": lambda r: SolutionBitmask().canVisitAllRooms([list(x) for x in r]),
}


def oracle_floyd(rooms):
    """Oracle độc lập: bao đóng bắc cầu bằng Floyd-Warshall rồi kiểm tra hàng của phòng 0."""
    n = len(rooms)
    reach = [[i == j for j in range(n)] for i in range(n)]
    for u, keys in enumerate(rooms):
        for k in keys:
            reach[u][k] = True
    for m in range(n):
        for i in range(n):
            if reach[i][m]:
                for j in range(n):
                    if reach[m][j]:
                        reach[i][j] = True
    return all(reach[0])


def check(rooms, want=None, names=None):
    if want is None:
        want = oracle_floyd(rooms)
    for name in (names or IMPLS):
        old = sys.getrecursionlimit()
        sys.setrecursionlimit(max(old, 5000))
        got = IMPLS[name](rooms)
        sys.setrecursionlimit(old)
        assert got is want, (name, rooms, got, want)


check(ex1, True)
check(ex2, False)
print("ví dụ 1, 2: cả", len(IMPLS), "cài đặt và oracle Floyd-Warshall đúng")

ví dụ 1, 2: cả 9 cài đặt và oracle Floyd-Warshall đúng


### B. Edge case

- `n = 2`: `[[1], []]` → `True`; `[[], [0]]` → `False` (cạnh một chiều); `[[0], []]` → `False` (chìa tự mở lại phòng 0).
- Phòng `0` không có chìa (`n ≥ 2`) → `False`.
- Mọi phòng có chìa mở mọi phòng (đồ thị đầy đủ) → `True`.
- Chu trình `0 → 1 → 2 → 0` → `True`; chu trình nhưng thiếu một phòng.
- Phòng có chìa **tự mở chính nó**.
- Chuỗi và chuỗi đảo (`1 → 0` không giúp).
- Phòng cuối chỉ mở được bởi chính nó.

In [19]:
cases = [
    ([[1], []], True), ([[], [0]], False), ([[0], []], False),
    ([[], [], []], False),
    ([list(range(1, 5))] + [[] for _ in range(4)], True),
    ([[1], [2], [0]], True), ([[1], [0], [2]], False),
    ([[0, 1], [1, 2], [2]], True),                                 # chìa tự mở chính phòng
    ([[1], [2], [3], [0]], True), ([[3], [0], [1], [2]], True),
    ([[1], [0, 2], [1], [3]], False),                                # phòng 3 chỉ mở bởi chính nó
    ([[2], [0], [1]], True),                                         # 0->2->1 (và 1->0)
]
for rooms, want in cases:
    check(rooms)
for rooms, want in cases:
    assert oracle_floyd(rooms) is want, (rooms, oracle_floyd(rooms), want)
print(len(cases), "edge case: tất cả cài đặt khớp oracle (đáp án viết tay cũng khớp oracle)")

12 edge case: tất cả cài đặt khớp oracle (đáp án viết tay cũng khớp oracle)


### C. Phản ví dụ: cài đặt sai phải bị bắt

In [20]:
def bug_union_find(rooms):                      # coi đồ thị là vô hướng
    n = len(rooms)
    parent = list(range(n))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    for u, keys in enumerate(rooms):
        for k in keys:
            parent[find(u)] = find(k)
    return len({find(i) for i in range(n)}) == 1


def bug_no_visited(rooms, guard=20000):          # quên visited: lặp vô hạn khi có chu trình
    queue = deque([0])
    pops = 0
    seen_any = {0}
    while queue:
        u = queue.popleft()
        pops += 1
        if pops > guard:
            raise RuntimeError("không kết thúc (chu trình)")
        for k in rooms[u]:
            seen_any.add(k)
            queue.append(k)
    return len(seen_any) == len(rooms)


def bug_count_n_minus_1(rooms):                  # đếm sai: không tính phòng 0 nhưng so với n
    visited = set()
    queue = deque([0])
    while queue:
        u = queue.popleft()
        for k in rooms[u]:
            if k not in visited and k != 0:
                visited.add(k)
                queue.append(k)
    return len(visited) == len(rooms)             # SAI: phải là n - 1


def bug_only_direct(rooms):                       # chỉ xét chìa trong phòng 0 (không lan tiếp)
    return len(set(rooms[0]) | {0}) == len(rooms)


BUGS = [bug_union_find, bug_no_visited, bug_count_n_minus_1, bug_only_direct]
rng = random.Random(1)
tests = []
for _ in range(600):
    n = rng.randint(2, 7)
    tests.append(random_rooms(rng, n, rng.randint(1, 2 * n)))
tests += [[[], [0]], [[1], [0]], [[1], [2], [0]]]
for bug in BUGS:
    bad = None
    for t in tests:
        try:
            got = bug(t)
        except RuntimeError:
            got = "RuntimeError"
        if got != oracle_floyd(t):
            bad = (t, got)
            break
    assert bad is not None, bug.__name__
    print(f"{bug.__name__:22s} sai trên {bad[0]}: ra {bad[1]}, đúng {oracle_floyd(bad[0])}")

bug_union_find         sai trên [[0, 1], [1], [1]]: ra True, đúng False
bug_no_visited         sai trên [[0, 1], [1], [1]]: ra RuntimeError, đúng False
bug_count_n_minus_1    sai trên [[3], [2, 3], [0, 4], [1, 4], []]: ra False, đúng True
bug_only_direct        sai trên [[3], [2, 3], [0, 4], [1, 4], []]: ra False, đúng True


### D. Vét cạn: mọi đồ thị có hướng nhỏ

Với `n` phòng, mỗi phòng có một tập chìa bất kỳ (kể cả chìa mở chính nó). Số đồ thị là `2^(n²)`: `n = 2` → 16, `n = 3` → 512, `n = 4` → 65 536 đồ thị. Mọi cài đặt phải khớp oracle Floyd–Warshall trên **tất cả**.

In [21]:
t0 = time.perf_counter()
total = 0
true_count = {}
for n in (2, 3, 4):
    cnt_true = 0
    for bits in product((0, 1), repeat=n * n):
        rooms = [[j for j in range(n) if bits[i * n + j]] for i in range(n)]
        want = oracle_floyd(rooms)
        for name in ("fixed_point", "dfs", "bfs", "stack", "bitmask"):
            assert IMPLS[name](rooms) is want, (name, rooms)
        total += 1
        cnt_true += want
    true_count[n] = cnt_true
print(f"mọi đồ thị có hướng n = 2, 3, 4 ({total} đồ thị): 5 cài đặt khớp Floyd-Warshall; số ca True theo n =", true_count, f"; {time.perf_counter() - t0:.1f} s")
assert true_count[2] == 8 and total == 16 + 512 + 65536

mọi đồ thị có hướng n = 2, 3, 4 (66064 đồ thị): 5 cài đặt khớp Floyd-Warshall; số ca True theo n = {2: 8, 3: 256, 4: 38912} ; 0.9 s


### E. Ngẫu nhiên có seed, đúng ràng buộc

Đồ thị ngẫu nhiên với `n` từ `2` đến `60`, mật độ cạnh khác nhau (gồm đồ thị **chắc chắn** vào được mọi phòng: chuỗi ngẫu nhiên cộng nhiễu).

In [22]:
rng = random.Random(20240841)
cnt = n_true = 0
for _ in range(2000):
    n = rng.randint(2, 60)
    kind = rng.random()
    if kind < 0.3:                                   # đảm bảo True: hoán vị ngẫu nhiên làm chuỗi + nhiễu
        order = list(range(1, n))
        rng.shuffle(order)
        order = [0] + order
        rooms = [set() for _ in range(n)]
        for a, b in zip(order, order[1:]):
            rooms[a].add(b)
        for _ in range(rng.randint(0, n)):
            rooms[rng.randrange(n)].add(rng.randrange(n))
        rooms = [sorted(s) for s in rooms]
    else:
        rooms = random_rooms(rng, n, rng.randint(1, 3 * n))
    want = oracle_floyd(rooms)
    check(rooms, want)
    cnt += 1
    n_true += want
print(cnt, "đồ thị ngẫu nhiên (True:", n_true, "False:", cnt - n_true, "): mọi cài đặt khớp oracle")

# tính chất: thêm cạnh không biến True thành False; bỏ phòng 0 khỏi mọi chìa của phòng khác không đổi kết quả
for _ in range(300):
    n = rng.randint(2, 12)
    rooms = random_rooms(rng, n, rng.randint(1, 2 * n))
    base = Solution().canVisitAllRooms([list(x) for x in rooms])
    u, k = rng.randrange(n), rng.randrange(n)
    more = [sorted(set(x) | ({k} if i == u else set())) for i, x in enumerate(rooms)]
    if base:
        assert Solution().canVisitAllRooms(more) is True
print("tính chất đơn điệu (thêm chìa không làm True thành False): OK")

2000 đồ thị ngẫu nhiên (True: 657 False: 1343 ): mọi cài đặt khớp oracle
tính chất đơn điệu (thêm chìa không làm True thành False): OK


### F. Giới hạn lớn nhất

`n = 1000`, `E ≤ 3000`: chuỗi (độ sâu 1000, ca xấu nhất cho DFS đệ quy), sao, chu trình lớn, đồ thị ngẫu nhiên, chuỗi bị cắt ở giữa.

In [23]:
n = 1000
cycle = [[(i + 1) % n] for i in range(n)]
broken = [[i + 1] for i in range(n - 1)] + [[]]
broken[500] = []                                     # cắt chuỗi giữa chừng
cases = {
    "chuỗi": ([[i + 1] for i in range(n - 1)] + [[]], True),
    "sao": ([list(range(1, n))] + [[] for _ in range(n - 1)], True),
    "chu trình 1000": (cycle, True),
    "chuỗi bị cắt": (broken, False),
    "ngẫu nhiên": (random_rooms(random.Random(5), n, 3000), None),
    "chuỗi ngược (0 không mở được gì)": ([[]] + [[i - 1] for i in range(1, n)], False),
}
for name, (g, want) in cases.items():
    if want is None:
        want = oracle_floyd(g) if False else Solution().canVisitAllRooms([list(x) for x in g])
    check(g, want, [k for k in IMPLS if k != "fixed_point"])
    print(f"{name:34s}: {want}")
assert sum(len(x) for x in cases["ngẫu nhiên"][0]) <= 3000

chuỗi                             : True
sao                               : True
chu trình 1000                    : True
chuỗi bị cắt                      : False
ngẫu nhiên                        : False
chuỗi ngược (0 không mở được gì)  : False


## 13. Các lỗi người mới hay gặp

### Lỗi 1: coi đồ thị là vô hướng (Union-Find)

Chìa chỉ mở cửa **một chiều**. `[[], [0]]`: phòng `1` có chìa mở phòng `0` nhưng ta **không** vào được phòng `1` vì phòng `0` không có chìa nào.

In [24]:
g = [[], [0]]
print("Union-Find:", bug_union_find(g), "| đúng:", Solution().canVisitAllRooms(g))
g = [[1], [0, 2], [], [2]]
print("Union-Find:", bug_union_find(g), "| đúng:", Solution().canVisitAllRooms(g))
assert bug_union_find([[], [0]]) is True and Solution().canVisitAllRooms([[], [0]]) is False

Union-Find: True | đúng: False
Union-Find: True | đúng: False


**Vì sao sai:** DSU gộp hai đỉnh khi có cạnh theo **bất kỳ** hướng; còn bài hỏi khả năng tới được theo **hướng** cạnh. **Đúng:** DFS/BFS từ `0` theo hướng `rooms[u]`.

### Lỗi 2: quên `visited` → lặp vô hạn khi có chu trình

Phòng `0` có chìa `1`, phòng `1` có chìa `0`: nếu không ghi nhớ thì hai phòng đẩy nhau mãi.

In [25]:
try:
    bug_no_visited([[1], [0]])
except RuntimeError as e:
    print("CODE SAI ->", e)
print("CODE ĐÚNG ->", Solution().canVisitAllRooms([[1], [0]]))

CODE SAI -> không kết thúc (chu trình)
CODE ĐÚNG -> True


**Vì sao sai:** không có gì chặn quay lại. **Đúng:** `visited` và chỉ đẩy phòng chưa thăm (đánh dấu ngay lúc đẩy).

### Lỗi 3: đếm sai `n − 1` thay vì `n`

Nếu không thêm phòng `0` vào `visited` (hoặc loại phòng `0` khi đẩy) thì đếm thiếu một phòng.

In [26]:
print("so với n:", bug_count_n_minus_1(ex1), "| đúng:", Solution().canVisitAllRooms(ex1))
assert bug_count_n_minus_1(ex1) is False and Solution().canVisitAllRooms(ex1) is True

so với n: False | đúng: True


**Vì sao sai:** phòng `0` cũng là một phòng đã vào. **Đúng:** khởi tạo `visited = {0}`.

### Lỗi 4: chỉ xét chìa trong phòng `0`

Chìa ở phòng `0` chỉ mở được các phòng kế tiếp; từ đó còn nhiều chìa khác. Phải **lan tiếp**.

In [27]:
print("chỉ phòng 0:", bug_only_direct(ex1), "| đúng:", Solution().canVisitAllRooms(ex1))
assert bug_only_direct(ex1) is False and Solution().canVisitAllRooms(ex1) is True

chỉ phòng 0: False | đúng: True


**Vì sao sai:** ví dụ 1 phải đi qua `0 → 1 → 2 → 3`. **Đúng:** duyệt đồ thị (BFS/DFS) cho tới khi không còn phòng mới.

### Lỗi 5: đánh dấu `visited` muộn (lúc lấy ra) — đúng kết quả nhưng lãng phí

Đánh dấu phòng là đã thăm **khi lấy ra** (không phải khi đẩy vào) vẫn cho đáp án đúng nếu bạn kiểm tra `visited` trước khi đẩy; nhưng một phòng có thể bị **đẩy nhiều lần** bởi nhiều phòng khác trước khi được lấy ra, nên hàng đợi dài hơn và có phòng bị xử lý lại. Đếm số lần lấy ra trên đồ thị đầy đủ:

In [28]:
def pops_count(rooms, mark_on_push):
    visited = {0} if mark_on_push else set()
    queue = deque([0])
    pops = 0
    while queue:
        u = queue.popleft()
        pops += 1
        if not mark_on_push:
            if u in visited and pops > 1:
                continue                        # đã xử lý rồi (bỏ qua bản trùng)
            visited.add(u)
        for k in rooms[u]:
            if k not in visited:
                if mark_on_push:
                    visited.add(k)
                queue.append(k)
    return pops


full = [list(range(60)) for _ in range(60)]
print("đồ thị đầy đủ 60 phòng: lấy ra", pops_count(full, True), "lần (đánh dấu khi đẩy) so với", pops_count(full, False), "lần (đánh dấu khi lấy ra)")
assert pops_count(full, True) == 60 and pops_count(full, False) > 60

đồ thị đầy đủ 60 phòng: lấy ra 60 lần (đánh dấu khi đẩy) so với 1771 lần (đánh dấu khi lấy ra)


**Vì sao lãng phí:** nhiều phòng cùng đẩy một phòng chưa thăm vào hàng đợi trước khi nó được xử lý. **Đúng:** `visited.add(k)` ngay khi `queue.append(k)`, để mỗi phòng vào hàng đợi tối đa một lần.

### Lỗi 6 (bonus): DFS đệ quy trên chuỗi 1000 phòng

Với giới hạn đệ quy mặc định, chuỗi `n = 1000` có thể vượt giới hạn (độ sâu ~1000 cộng khung gọi). BFS không gặp vấn đề này.

In [29]:
chain = [[i + 1] for i in range(999)] + [[]]
old = sys.getrecursionlimit()
sys.setrecursionlimit(500)
try:
    can_visit_dfs(chain)
    print("không lỗi")
except RecursionError:
    print("can_visit_dfs + giới hạn 500 -> RecursionError (độ sâu ~1000)")
finally:
    sys.setrecursionlimit(old)
print("BFS cùng input:", can_visit_bfs(chain))

can_visit_dfs + giới hạn 500 -> RecursionError (độ sâu ~1000)
BFS cùng input: True


## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "bắt đầu từ một nơi, mỗi bước mở thêm các nơi mới theo quy tắc, hỏi có tới hết không" → **duyệt đồ thị (DFS/BFS) từ nguồn + tập `visited`**.

**Mẹo:**

1. Nhận ra **danh sách kề** ngay trong đầu vào (`rooms[i]` = các đỉnh kề của `i`).
2. Xác định đồ thị **có hướng hay vô hướng** — quyết định cấu trúc dữ liệu (DSU chỉ cho vô hướng).
3. Luôn có `visited`; **đánh dấu lúc đưa vào** hàng đợi/ngăn xếp.
4. Đáp án "tới hết": `len(visited) == n` (nhớ phòng `0` ban đầu).
5. Giới hạn `n ≈ 10^3` với DFS đệ quy: để ý độ sâu; BFS an toàn hơn.
6. Oracle độc lập: **bao đóng bắc cầu** (Floyd–Warshall) cho đồ thị nhỏ.

**Họ bài cùng pattern:** Number of Provinces (547, vô hướng), Find if Path Exists in Graph (1971), All Paths From Source to Target (797), Course Schedule (207, phát hiện chu trình trên đồ thị có hướng), Reachable Nodes With Restrictions (2368).

## 15. Biến thể

1. **Đếm số phòng vào được** (bài tập 5).
2. **Liệt kê các phòng không vào được.**
3. **Có thể vào mọi phòng rồi quay lại phòng `0` không?** (đồ thị liên thông mạnh qua `0`: cần cả thuận lẫn nghịch).
4. **Số chìa tối thiểu thêm vào** để vào được mọi phòng.
5. **Thứ tự vào phòng** (một trật tự hợp lệ).

Cài đặt biến thể 2 và 3:

In [30]:
def unreachable_rooms(rooms):
    visited = {0}
    q = deque([0])
    while q:
        u = q.popleft()
        for k in rooms[u]:
            if k not in visited:
                visited.add(k)
                q.append(k)
    return sorted(set(range(len(rooms))) - visited)


def can_visit_all_and_return(rooms):
    """Biến thể 3: vào được mọi phòng VÀ từ mỗi phòng đều quay lại được phòng 0 (liên thông mạnh theo 0)."""
    n = len(rooms)
    if unreachable_rooms(rooms):
        return False
    rev = [[] for _ in range(n)]
    for u, keys in enumerate(rooms):
        for k in keys:
            rev[k].append(u)                         # đảo hướng mọi cạnh
    return not unreachable_rooms(rev)


print(unreachable_rooms(ex2), unreachable_rooms(ex1))
assert unreachable_rooms(ex2) == [2] and unreachable_rooms(ex1) == []
print("vào hết và quay lại 0:", can_visit_all_and_return([[1], [2], [0]]), can_visit_all_and_return(ex1), can_visit_all_and_return([[1, 2], [], [0]]))
assert can_visit_all_and_return([[1], [2], [0]]) is True and can_visit_all_and_return(ex1) is False
assert can_visit_all_and_return([[1, 2], [], [0]]) is False

[2] []
vào hết và quay lại 0: True False False


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Lặp tới điểm bất động | Mỗi vòng thêm mọi phòng mở được | `O(n·(n+E))` | `O(n)` | Oracle đơn giản |
| 2. DFS đệ quy | Vào phòng, đệ quy theo mỗi chìa | `O(n+E)` | `O(n)` call stack | Ngắn; nâng `setrecursionlimit` |
| 3. BFS / DFS ngăn xếp | Hàng đợi, đánh dấu khi đưa vào | `O(n+E)` | `O(n)` | **Đáp án chính**, an toàn |
| 4. Bao đóng bitmask | Tập đỉnh là số nguyên, hợp bằng `|` | `O(n²/64 + E)` | `O(n²/64)` | Muốn bao đóng của nhiều nguồn một lúc |

Cả bốn khớp Floyd–Warshall trên 66 064 đồ thị vét cạn và hàng nghìn đồ thị ngẫu nhiên (mục 12).

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi mới chạy:

In [31]:
r = [[2, 3], [], [1], [0]]
print(Solution().canVisitAllRooms(r))
print(unreachable_rooms([[1], [], [0]]))
print(Solution().canVisitAllRooms([[1], [2], [0]]), Solution().canVisitAllRooms([[], [0]]))

True
[2]
True False


### Bài 2 (tính tay)

`rooms = [[1, 2], [3], [3], [4], []]`.

- (a) Vẽ các cạnh có hướng và nêu thứ tự BFS bắt đầu từ `0`.
- (b) Có vào được mọi phòng không?
- (c) Nếu đổi `rooms[3]` thành `[]` thì kết quả thay đổi thế nào và vì sao?

Tự giải rồi kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện BFS:

In [32]:
def bfs_todo(rooms):
    visited = {0}
    queue = deque([0])
    while queue:
        u = queue.popleft()
        for k in rooms[u]:
            pass            # TODO: nếu k chưa thăm thì đánh dấu ngay và đưa vào hàng đợi
    return None             # TODO: trả len(visited) == len(rooms)


# Khi làm xong, bỏ comment để kiểm tra:
# assert bfs_todo([[1], [2], [3], []]) is True and bfs_todo([[1, 3], [3, 0, 1], [2], [0]]) is False

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết `class MySolution` bằng DFS đệ quy (nhớ `visited`):

In [33]:
class MySolution:
    def canVisitAllRooms(self, rooms):
        pass            # TODO


# Khi làm xong:
# assert MySolution().canVisitAllRooms([[1], [2], [3], []]) is True
# assert MySolution().canVisitAllRooms([[1, 3], [3, 0, 1], [2], [0]]) is False

### Bài 5 (mini challenge): thứ tự vào phòng

Viết `visit_order(rooms)` trả về `(thứ tự các phòng đã vào theo BFS, số phòng không vào được)`. Thứ tự là một dãy các phòng bắt đầu từ `0`, mỗi phòng (trừ `0`) có chìa nằm ở một phòng **đứng trước** nó trong dãy. Kiểm bằng hàm hợp lệ (không so cứng thứ tự).

In [34]:
def visit_order(rooms):
    pass            # TODO


# Khi làm xong (kiểm bằng hàm hợp lệ):
# order, stuck = visit_order([[1, 2], [3], [3], [4], []])
# assert order[0] == 0 and stuck == 0 and sorted(order) == [0, 1, 2, 3, 4]

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: vẽ cạnh `u → k` cho mỗi `k ∈ rooms[u]`.
- Mức 2: từ `0` có chìa `2, 3`; phòng `2` có chìa `1`; phòng `3` có chìa `0`.
- Mức 3: `True` (đi tới `1, 2, 3`); `[[1],[],[0]]`: phòng `2` không có chìa nào mở được nó (chỉ `rooms[2]` chứa `0`) → `[2]`; chu trình `[[1],[2],[0]]` → `True`; `[[], [0]]` → `False`.

**Bài 2**

- Mức 1: cạnh `0→1, 0→2, 1→3, 2→3, 3→4`.
- Mức 2: BFS: `0`, `1`, `2`, `3`, `4`.
- Mức 3: (b) có; (c) nếu `rooms[3] = []` thì phòng `4` không còn chìa nào mở nó → `False`.

**Bài 3**

- Mức 1: giống Cách 3.
- Mức 2: `if k not in visited: visited.add(k); queue.append(k)`.
- Mức 3: cuối `return len(visited) == len(rooms)`.

**Bài 4**

- Mức 1: `visited = set()`, `dfs(u)`.
- Mức 2: bỏ qua nếu `u in visited`.
- Mức 3: `dfs(0)`; trả `len(visited) == len(rooms)`; nhớ `setrecursionlimit`.

**Bài 5**

- Mức 1: BFS ghi lại thứ tự lấy ra.
- Mức 2: số phòng không vào được là `n − len(order)`.
- Mức 3: kiểm tra thứ tự: mỗi phòng sau `0` có một phòng đứng trước chứa chìa của nó.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert` để chạy kiểm tra.

### Bài 1

In [35]:
assert Solution().canVisitAllRooms([[2, 3], [], [1], [0]]) is True
assert unreachable_rooms([[1], [], [0]]) == [2]
assert Solution().canVisitAllRooms([[1], [2], [0]]) is True and Solution().canVisitAllRooms([[], [0]]) is False
print("Bài 1 OK")

Bài 1 OK


### Bài 2

(a) Cạnh `0→1, 0→2, 1→3, 2→3, 3→4`; BFS: `0, 1, 2, 3, 4`. (b) Vào được hết (`True`). (c) Nếu `rooms[3] = []` thì không còn chìa mở phòng `4` → `False`.

In [36]:
r = [[1, 2], [3], [3], [4], []]
order = []
visited = {0}
q = deque([0])
while q:
    u = q.popleft()
    order.append(u)
    for k in r[u]:
        if k not in visited:
            visited.add(k)
            q.append(k)
assert order == [0, 1, 2, 3, 4]
assert Solution().canVisitAllRooms(r) is True
r2 = [[1, 2], [3], [3], [], []]
assert Solution().canVisitAllRooms(r2) is False and oracle_floyd(r2) is False
print("Bài 2 OK")

Bài 2 OK


### Bài 3

In [37]:
def bfs_answer(rooms):
    visited = {0}
    queue = deque([0])
    while queue:
        u = queue.popleft()
        for k in rooms[u]:
            if k not in visited:
                visited.add(k)
                queue.append(k)
    return len(visited) == len(rooms)


assert bfs_answer([[1], [2], [3], []]) is True and bfs_answer([[1, 3], [3, 0, 1], [2], [0]]) is False
rng = random.Random(3)
for _ in range(300):
    g = random_rooms(rng, rng.randint(2, 8), rng.randint(1, 12))
    assert bfs_answer(g) is oracle_floyd(g)
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [38]:
class MySolutionAnswer:
    def canVisitAllRooms(self, rooms):
        sys.setrecursionlimit(max(sys.getrecursionlimit(), 5000))
        visited = set()

        def dfs(u):
            if u in visited:
                return
            visited.add(u)
            for k in rooms[u]:
                dfs(k)

        dfs(0)
        return len(visited) == len(rooms)


assert MySolutionAnswer().canVisitAllRooms([[1], [2], [3], []]) is True
assert MySolutionAnswer().canVisitAllRooms([[1, 3], [3, 0, 1], [2], [0]]) is False
rng = random.Random(4)
for _ in range(300):
    g = random_rooms(rng, rng.randint(2, 8), rng.randint(1, 12))
    assert MySolutionAnswer().canVisitAllRooms(g) is oracle_floyd(g)
print("Bài 4 OK")

Bài 4 OK


### Bài 5

In [39]:
def visit_order_answer(rooms):
    visited = {0}
    q = deque([0])
    order = []
    while q:
        u = q.popleft()
        order.append(u)
        for k in rooms[u]:
            if k not in visited:
                visited.add(k)
                q.append(k)
    return order, len(rooms) - len(order)


def valid_order(rooms, order):
    if not order or order[0] != 0 or len(set(order)) != len(order):
        return False
    seen_keys = set(rooms[0])
    for room in order[1:]:
        if room not in seen_keys:
            return False
        seen_keys |= set(rooms[room])
    return True


order, stuck = visit_order_answer([[1, 2], [3], [3], [4], []])
assert order[0] == 0 and stuck == 0 and sorted(order) == [0, 1, 2, 3, 4] and valid_order([[1, 2], [3], [3], [4], []], order)
rng = random.Random(5)
for _ in range(300):
    g = random_rooms(rng, rng.randint(2, 9), rng.randint(1, 14))
    order, stuck = visit_order_answer(g)
    assert valid_order(g, order) and (stuck == 0) is oracle_floyd(g)
print("Bài 5 OK: thứ tự vào phòng hợp lệ và số phòng kẹt khớp oracle")

Bài 5 OK: thứ tự vào phòng hợp lệ và số phòng kẹt khớp oracle


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (tới được mọi đỉnh từ nguồn trên đồ thị có hướng, BFS):**

```python
class Solution:
    def canVisitAllRooms(self, rooms):
        visited = {0}
        queue = deque([0])
        while queue:
            u = queue.popleft()
            for k in rooms[u]:
                if k not in visited:
                    visited.add(k)           # đánh dấu ngay khi đẩy
                    queue.append(k)
        return len(visited) == len(rooms)
```

**Checklist trước khi submit:**

- [ ] Nhận ra `rooms` là **danh sách kề của đồ thị có hướng**; không dùng Union-Find (vô hướng)?
- [ ] Có `visited`, khởi tạo với phòng `0`, đánh dấu **ngay khi đưa vào** hàng đợi?
- [ ] Đáp án so với `n` (số phòng), không phải `n - 1`?
- [ ] DFS đệ quy: đã nâng `setrecursionlimit` hoặc dùng BFS (chuỗi 1000 phòng)?
- [ ] Đã thử `[[1],[2],[3],[]]` (True), `[[1,3],[3,0,1],[2],[0]]` (False), `[[], [0]]` (False), chu trình?
- [ ] Đúng tên `class Solution`, `canVisitAllRooms(self, rooms)`, không in debug.